# Week 1: Data Collection, EDA & Feature Engineering


This notebook covers the first week of the PM2.5 forecasting project, focusing on data loading, exploratory data analysis (EDA), and feature engineering (lags, rolling averages, and target creation).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

# Placeholder for loading actual data
# df = pd.read_csv('cairo_aqi_data.csv', parse_dates=['date'])

In [ ]:
# EDA: Visualizing Seasonal and District Patterns
def plot_eda(df):
    # 1. Seasonal Patterns (Checking Winter & Khamsin Spikes)
    plt.figure(figsize=(10, 5))
    sns.boxplot(data=df, x='season', y='pm2_5', palette='Set2')
    plt.title('PM2.5 Distribution by Season', weight='bold')
    plt.show()

    # 2. District Volatility (Highlighting 6th of October)
    plt.figure(figsize=(10, 5))
    sns.boxplot(data=df, x='district', y='pm2_5', palette='Set3')
    plt.title('PM2.5 Distribution by District', weight='bold')
    plt.xticks(rotation=45)
    plt.show()

In [ ]:
# Feature Engineering: Lags, Rolling Windows, and Targets
def create_features(df):
    df = df.sort_values(by=['district', 'date'])

    # Lags
    df['pm2_5_lag_1'] = df.groupby('district')['pm2_5'].shift(1)
    df['pm2_5_lag_2'] = df.groupby('district')['pm2_5'].shift(2)

    # Rolling features (e.g., 3-day average)
    df['pm2_5_rolling_mean_3d'] = df.groupby('district')['pm2_5'].shift(1).rolling(window=3).mean()

    # Targets (t+1, t+2, t+3)
    df['target_1d'] = df.groupby('district')['pm2_5'].shift(-1)
    df['target_2d'] = df.groupby('district')['pm2_5'].shift(-2)
    df['target_3d'] = df.groupby('district')['pm2_5'].shift(-3)

    # Classification Target (Health Risk Threshold > 35)
    df['target_risk_1d'] = (df['target_1d'] > 35.0).astype(int)

    return df.dropna()